# Exercise 3 (Starter) — Build, Cross-Validate, Test & Serialize an End-to-End Pipeline

**Timebox:** 25 minutes
**Goal:** Assemble a complete `sklearn.pipeline.Pipeline` that takes raw StreamPulse columns, applies preprocessing + a classifier, evaluates with 5-fold Stratified CV and held-out test metrics, saves the pipeline to disk with `joblib`, and passes a reload-and-predict smoke test!

### Your Tasks
1. **Task 1 (`create_churn_pipeline`)**: Build a `Pipeline` with two steps:
   - `"preprocessor"`: `ColumnTransformer` (median impute + `StandardScaler` for `NUMERIC_COLS`; most_frequent impute + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` for `CATEGORICAL_COLS`)
   - `"classifier"`: a `RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=42)` (or `LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)`)
2. **Task 2 (`train_and_evaluate`)**:
   - Compute 5-fold Stratified CV `roc_auc` on `(X_train, y_train)`
   - Fit the pipeline on `(X_train, y_train)`
   - Evaluate `roc_auc` and `f1` on `(X_test, y_test)`
   - Return `(fitted_pipeline, metrics_dict)`
3. **Task 3**: Save the fitted pipeline to `artifacts/ex3_pipeline.joblib`, reload it, and verify predictions match the in-memory pipeline!


In [ ]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prepare clean data + domain features
cust = pd.read_csv("data/customers_raw.csv")
with open("data/usage_events.json", "r", encoding="utf-8") as f:
    usage = pd.DataFrame(json.load(f))
cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df = cust.drop_duplicates(subset=["customer_id"], keep="first").merge(usage, on="customer_id", how="left")
df["charges_per_login"] = df["monthly_charges"] / (df["login_days_30d"] + 1.0)
df["tickets_per_tenure_month"] = df["support_tickets_30d"] / (df["tenure_months"] + 1.0)

NUMERIC_COLS = [
    "age", "tenure_months", "monthly_charges",
    "login_days_30d", "watch_hours_30d", "support_tickets_30d",
    "auto_pay", "charges_per_login", "tickets_per_tenure_month",
]
CATEGORICAL_COLS = ["plan_tier", "country", "payment_method", "last_active_device"]

X = df[NUMERIC_COLS + CATEGORICAL_COLS]
y = df["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Ready! Train={len(X_train)}, Test={len(X_test)}")


In [ ]:
# ============================================================================
# TASK 1: Build the Pipeline
# ============================================================================
def create_churn_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    """Return an unfitted sklearn Pipeline(preprocessor -> classifier)."""
    # TODO 1: Build ColumnTransformer + Classifier inside a Pipeline
    raise NotImplementedError("Complete Task 1 inside create_churn_pipeline()")


In [ ]:
# ============================================================================
# TASK 2: Cross-validate on train, fit on full train, evaluate on test
# ============================================================================
def train_and_evaluate(
    pipeline: Pipeline,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> tuple[Pipeline, dict]:
    """Run 5-fold CV, fit on X_train, score on X_test, return (pipeline, metrics)."""
    # TODO 2a: Create StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # TODO 2b: Compute cv_roc_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    # TODO 2c: Fit pipeline on (X_train, y_train)
    # TODO 2d: Compute test_roc_auc and test_f1 on (X_test, y_test)
    # TODO 2e: Return pipeline, {"cv_roc_auc": ..., "test_roc_auc": ..., "test_f1": ...}
    raise NotImplementedError("Complete Task 2 inside train_and_evaluate()")


In [ ]:
# ============================================================================
# TASK 3 & SELF-CHECK TESTS
# ============================================================================
pipe = create_churn_pipeline(NUMERIC_COLS, CATEGORICAL_COLS)
fitted_pipe, metrics = train_and_evaluate(pipe, X_train, y_train, X_test, y_test)
print("Evaluation metrics:", metrics)

# Save & reload artifact
save_path = Path("artifacts/ex3_pipeline.joblib")
save_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(fitted_pipe, save_path)
reloaded_pipe = joblib.load(save_path)

# Automated self-checks
assert isinstance(fitted_pipe, Pipeline), "Must return a sklearn Pipeline!"
assert {"cv_roc_auc", "test_roc_auc", "test_f1"}.issubset(metrics.keys())
assert metrics["cv_roc_auc"] >= 0.72, f"Expected cv_roc_auc >= 0.72, got {metrics['cv_roc_auc']:.3f}"
assert metrics["test_roc_auc"] >= 0.72, f"Expected test_roc_auc >= 0.72, got {metrics['test_roc_auc']:.3f}"
assert save_path.exists(), "Saved joblib file not found!"

orig_probs = fitted_pipe.predict_proba(X_test)[:, 1]
reloaded_probs = reloaded_pipe.predict_proba(X_test)[:, 1]
assert np.allclose(orig_probs, reloaded_probs), "Reloaded pipeline predictions differ from original!"

print("\n[ALL TESTS PASSED] Pipeline trained, validated, saved, and reloaded identically!")


### Bonus Stretch Task
Test what happens if you tune the classification threshold from `0.50` down to `0.35` on `y_prob`: how do **Precision** and **Recall** change, and when would StreamPulse's retention team prefer higher recall?
